# ResearchLanka AI Relevance Model Selection

This notebook trains several local text classifiers from the 5k Gemini/OpenRouter AI relevance labels, chooses the best model on a locked validation set, and reports the final score once on a locked test set.

No OpenRouter/API key is needed. The labelled train/validation/test CSVs are already split before modelling so the model never sees validation/test rows during training.

In [ ]:
from pathlib import Path
import json
import math
import shutil

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
OUTPUT_DIR = Path('/kaggle/working/ai_relevance_model_selection_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COLUMNS = [
    'title', 'abstract', 'keywords', 'topics', 'concepts',
    'primary_topic', 'primary_subfield', 'primary_field', 'primary_domain',
]
ID_COLUMNS = ['record_number', 'publication_id', 'openalex_id', 'doi', 'source_record_id']
METADATA_COLUMNS = [
    'source_dataset', 'source_institution_id', 'source_record_id',
    'openalex_id', 'doi', 'title', 'publication_date', 'publication_year',
    'primary_topic', 'primary_subfield', 'primary_field', 'primary_domain',
]

print('Output directory:', OUTPUT_DIR)

## Locate Kaggle Dataset Files

Upload `kaggle_ai_relevance_model_selection_dataset.zip` as a Kaggle dataset. Kaggle unzips it under `/kaggle/input/<dataset-name>/`.

In [ ]:
def find_input_file(filename: str) -> Path:
    matches = list(Path('/kaggle/input').rglob(filename))
    if not matches:
        raise FileNotFoundError(f'Could not find {filename} under /kaggle/input')
    if len(matches) > 1:
        print(f'Found multiple matches for {filename}; using {matches[0]}')
    return matches[0]

FULL_LABELLED_PATH = find_input_file('ai_llm_5000_predictions_openrouter_gemini_3_8_flash.csv')
TRAIN_PATH = find_input_file('ai_relevance_train_labels.csv')
VALIDATION_PATH = find_input_file('ai_relevance_validation_labels.csv')
TEST_PATH = find_input_file('ai_relevance_test_labels.csv')
CORPUS_PATH = find_input_file('common_publications_final.csv')

print('Full 5k labels:', FULL_LABELLED_PATH)
print('Train:', TRAIN_PATH)
print('Validation:', VALIDATION_PATH)
print('Test:', TEST_PATH)
print('Full corpus:', CORPUS_PATH)

## Helper Functions

In [ ]:
def clean_value(value) -> str:
    if value is None or pd.isna(value):
        return ''
    text = str(value).strip()
    if text.casefold() in {'', 'nan', 'none', 'null'}:
        return ''
    return text

def publication_id_from_row(row, fallback) -> str:
    for col in ID_COLUMNS:
        if col in row.index:
            value = clean_value(row[col])
            if value:
                return value
    return f'row-{fallback}'

def combined_text(frame: pd.DataFrame, text_columns=TEXT_COLUMNS) -> pd.Series:
    available = [col for col in text_columns if col in frame.columns]
    if not available:
        return pd.Series('', index=frame.index)
    text = frame[available].fillna('').astype(str).agg(' '.join, axis=1)
    text = text.str.replace(r'\s+', ' ', regex=True).str.strip()
    text = text.replace({'nan': '', 'None': '', 'null': ''})
    return text

def load_split(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    frame['publication_id'] = frame.apply(lambda row: publication_id_from_row(row, row.name), axis=1)
    frame['text'] = combined_text(frame)
    frame = frame[
        frame['ai_llm_status'].eq('success')
        & frame['ai_llm_label'].isin(['AI', 'NON_AI'])
        & (frame['text'] != '')
    ].copy()
    return frame

def sigmoid_abs_margin(margin: float) -> float:
    return 1.0 / (1.0 + math.exp(-abs(float(margin))))

def decision_margins(model, text):
    if hasattr(model, 'decision_function'):
        raw = model.decision_function(text)
        if getattr(raw, 'ndim', 1) == 1:
            return np.asarray(raw, dtype=float)
        return np.asarray([row[np.argmax(np.abs(row))] for row in raw], dtype=float)
    if hasattr(model, 'predict_proba'):
        probs = model.predict_proba(text)
        return np.asarray(probs.max(axis=1) - 0.5, dtype=float)
    return np.zeros(len(text), dtype=float)

def evaluate_predictions(y_true, y_pred) -> dict:
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=['AI', 'NON_AI'], zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'ai_precision': float(precision[0]),
        'ai_recall': float(recall[0]),
        'ai_f1': float(f1[0]),
        'non_ai_precision': float(precision[1]),
        'non_ai_recall': float(recall[1]),
        'non_ai_f1': float(f1[1]),
    }

def make_pipeline(vectorizer_name: str, classifier):
    if vectorizer_name == 'word_1_2':
        vectorizer = TfidfVectorizer(strip_accents='unicode', lowercase=True, ngram_range=(1, 2), min_df=2, max_df=0.95, max_features=50_000, sublinear_tf=True)
    elif vectorizer_name == 'word_1_3':
        vectorizer = TfidfVectorizer(strip_accents='unicode', lowercase=True, ngram_range=(1, 3), min_df=2, max_df=0.95, max_features=50_000, sublinear_tf=True)
    elif vectorizer_name == 'char_wb_3_5':
        vectorizer = TfidfVectorizer(strip_accents='unicode', lowercase=True, analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_df=0.95, max_features=75_000, sublinear_tf=True)
    else:
        raise ValueError(vectorizer_name)
    return Pipeline([('tfidf', vectorizer), ('clf', classifier)])

## Load Locked Splits

The validation and test splits are loaded only for evaluation. Candidate models are fitted on `train_df` only.

In [ ]:
train_df = load_split(TRAIN_PATH)
validation_df = load_split(VALIDATION_PATH)
test_df = load_split(TEST_PATH)
full_labelled = pd.read_csv(FULL_LABELLED_PATH, dtype=str, keep_default_na=False)
full_labelled['publication_id'] = full_labelled.apply(lambda row: publication_id_from_row(row, row.name), axis=1)

print('Train rows:', len(train_df), train_df['ai_llm_label'].value_counts().to_dict())
print('Validation rows:', len(validation_df), validation_df['ai_llm_label'].value_counts().to_dict())
print('Test rows:', len(test_df), test_df['ai_llm_label'].value_counts().to_dict())

split_overlap = (
    set(train_df['publication_id']) & set(validation_df['publication_id'])
    | set(train_df['publication_id']) & set(test_df['publication_id'])
    | set(validation_df['publication_id']) & set(test_df['publication_id'])
)
assert not split_overlap, f'Split leakage detected: {len(split_overlap)} overlapping IDs'
print('Split leakage check passed.')

## Define Candidate Models

These are practical sklearn text-classification baselines for a 5k labelled dataset:

- majority-class dummy baseline
- Logistic Regression
- Linear SVM
- calibrated Linear SVM
- SGD linear classifier
- Ridge classifier
- Multinomial Naive Bayes
- Complement Naive Bayes
- word TF-IDF and character TF-IDF variants

In [ ]:
candidate_specs = []

candidate_specs.append(('dummy_most_frequent', 'word_1_2', DummyClassifier(strategy='most_frequent')))

for C in [0.3, 1.0, 3.0, 10.0]:
    candidate_specs.append((f'logreg_word12_C{C}', 'word_1_2', LogisticRegression(C=C, class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)))
    candidate_specs.append((f'linearsvc_word13_C{C}', 'word_1_3', LinearSVC(C=C, class_weight='balanced', max_iter=5000, random_state=RANDOM_STATE, dual='auto')))

for alpha in [1e-5, 1e-4, 1e-3]:
    candidate_specs.append((f'sgd_logloss_word13_alpha{alpha}', 'word_1_3', SGDClassifier(loss='log_loss', alpha=alpha, class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)))
    candidate_specs.append((f'ridge_word13_alpha{alpha}', 'word_1_3', RidgeClassifier(alpha=alpha, class_weight='balanced', random_state=RANDOM_STATE)))

for alpha in [0.1, 0.5, 1.0]:
    candidate_specs.append((f'multinomial_nb_word12_alpha{alpha}', 'word_1_2', MultinomialNB(alpha=alpha)))
    candidate_specs.append((f'complement_nb_word12_alpha{alpha}', 'word_1_2', ComplementNB(alpha=alpha)))

for C in [1.0, 3.0]:
    candidate_specs.append((f'linearsvc_charwb35_C{C}', 'char_wb_3_5', LinearSVC(C=C, class_weight='balanced', max_iter=5000, random_state=RANDOM_STATE, dual='auto')))

print('Candidate models:', len(candidate_specs))
for name, vectorizer_name, _ in candidate_specs:
    print('-', name, '/', vectorizer_name)

## Train on Train Split and Select by Validation Macro F1

In [ ]:
results = []
fitted_models = {}

X_train = train_df['text']
y_train = train_df['ai_llm_label']
X_val = validation_df['text']
y_val = validation_df['ai_llm_label']

for name, vectorizer_name, classifier in candidate_specs:
    model = make_pipeline(vectorizer_name, clone(classifier))
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    metrics = evaluate_predictions(y_val, val_pred)
    row = {'model_name': name, 'vectorizer': vectorizer_name, **metrics}
    results.append(row)
    fitted_models[name] = model
    print(f"{name:40s} macro_f1={metrics['macro_f1']:.4f} ai_f1={metrics['ai_f1']:.4f} acc={metrics['accuracy']:.4f}")

leaderboard = pd.DataFrame(results).sort_values(
    ['macro_f1', 'ai_f1', 'accuracy'], ascending=False
).reset_index(drop=True)
leaderboard_path = OUTPUT_DIR / 'model_validation_leaderboard.csv'
leaderboard.to_csv(leaderboard_path, index=False)
leaderboard.head(20)

## Final Test Evaluation of the Best Validation Model

Only the single best validation model is evaluated on the test set.

In [ ]:
best_name = leaderboard.loc[0, 'model_name']
best_model = fitted_models[best_name]

test_pred = best_model.predict(test_df['text'])
test_metrics = evaluate_predictions(test_df['ai_llm_label'], test_pred)
test_report = classification_report(test_df['ai_llm_label'], test_pred, labels=['AI', 'NON_AI'], zero_division=0)
test_cm = confusion_matrix(test_df['ai_llm_label'], test_pred, labels=['AI', 'NON_AI'])

print('Best model:', best_name)
print(json.dumps(test_metrics, indent=2))
print(test_report)
print('Confusion matrix labels: AI, NON_AI')
print(test_cm)

pd.DataFrame(test_cm, index=['actual_AI', 'actual_NON_AI'], columns=['pred_AI', 'pred_NON_AI']).to_csv(OUTPUT_DIR / 'best_model_test_confusion_matrix.csv')

test_predictions = test_df[['publication_id', 'ai_llm_label', 'title', 'text']].copy()
test_predictions['predicted_label'] = test_pred
test_predictions['correct'] = test_predictions['ai_llm_label'].eq(test_predictions['predicted_label'])
test_predictions.to_csv(OUTPUT_DIR / 'best_model_test_predictions.csv', index=False)

## Refit Best Model on Train + Validation, Then Predict the Rest

The final model uses train + validation labels after model selection. The test set remains untouched and is used only for the final report above.

In [ ]:
best_spec = next(spec for spec in candidate_specs if spec[0] == best_name)
_, best_vectorizer_name, best_classifier = best_spec
final_model = make_pipeline(best_vectorizer_name, clone(best_classifier))
train_val_df = pd.concat([train_df, validation_df], ignore_index=True)
final_model.fit(train_val_df['text'], train_val_df['ai_llm_label'])

model_path = OUTPUT_DIR / 'best_ai_relevance_model.joblib'
joblib.dump(final_model, model_path)

model_summary = {
    'best_model_name': best_name,
    'best_vectorizer': best_vectorizer_name,
    'selection_metric': 'validation_macro_f1',
    'validation_leaderboard_top': leaderboard.head(10).to_dict(orient='records'),
    'test_metrics': test_metrics,
    'train_rows': int(len(train_df)),
    'validation_rows': int(len(validation_df)),
    'test_rows': int(len(test_df)),
    'final_train_plus_validation_rows': int(len(train_val_df)),
}
(OUTPUT_DIR / 'best_model_summary.json').write_text(json.dumps(model_summary, indent=2), encoding='utf-8')
print('Saved final model:', model_path)
print(json.dumps(model_summary, indent=2)[:2000])

## Predict Unlabelled/Rest Corpus

In [ ]:
corpus = pd.read_csv(CORPUS_PATH, dtype=str, keep_default_na=False)
corpus['source_row'] = corpus.index
corpus['publication_id'] = corpus.apply(lambda row: publication_id_from_row(row, row.name), axis=1)
corpus['text'] = combined_text(corpus)

labelled_ids = set(full_labelled['publication_id'].astype(str))
rest = corpus[~corpus['publication_id'].isin(labelled_ids)].copy()
rest = rest[rest['text'] != ''].copy()

rest_labels = final_model.predict(rest['text'])
rest_margins = decision_margins(final_model, rest['text'])

present_metadata = [col for col in METADATA_COLUMNS if col in rest.columns]
rest_predictions = rest[['source_row', 'publication_id', *present_metadata, 'text']].copy()
rest_predictions.insert(2, 'ai_model_label', rest_labels)
rest_predictions.insert(3, 'ai_model_confidence', [sigmoid_abs_margin(m) for m in rest_margins])
rest_predictions.insert(4, 'ai_model_margin', rest_margins)
rest_predictions.insert(5, 'selected_model', best_name)

prediction_path = OUTPUT_DIR / 'ai_relevance_best_model_rest_predictions.csv'
rest_predictions.to_csv(prediction_path, index=False)

prediction_summary = {
    'full_corpus_rows': int(len(corpus)),
    'excluded_5k_labelled_rows': int(len(corpus) - len(rest)),
    'predicted_rest_rows': int(len(rest_predictions)),
    'prediction_counts': {k: int(v) for k, v in rest_predictions['ai_model_label'].value_counts().items()},
    'selected_model': best_name,
}
(OUTPUT_DIR / 'rest_prediction_summary.json').write_text(json.dumps(prediction_summary, indent=2), encoding='utf-8')
print(json.dumps(prediction_summary, indent=2))
print('Saved predictions:', prediction_path)

## Zip Outputs for Download

In [ ]:
zip_base = Path('/kaggle/working/ai_relevance_model_selection_outputs')
zip_path = shutil.make_archive(str(zip_base), 'zip', OUTPUT_DIR)
print('Download:', zip_path)
print('Output files:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(' -', path.name)